In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd

In [4]:
from pathlib import Path


PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print(PROJECT_ROOT)

c:\Users\eademola\pd-risk-prediction


In [5]:
model_cohort = pd.read_csv(PROJECT_ROOT / "data/processed/cohort/ppmi_risk_cohort_multimodal_cdmgb.csv")
events = pd.read_csv(PROJECT_ROOT / "data/processed/outcomes/ppmi_outcomes.csv")
events

,PATNO,pd_event,pd_event_visit,pd_event_date,msa_event,msa_event_visit,msa_event_date,dlb_event,dlb_event_visit,dlb_event_date,n_visits,baseline_date,last_visit,last_visit_date,follow_up_years,death_event,death_date,death_timing,pd_case_type,pd_n_visits
0,3000,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,9,2011-02-01,V17,2021-05-01,10.245038,True,2026-03-01,death_without_pd,no_pd,0
1,3004,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,12,2011-04-01,V22,2026-05-01,15.082820,False,NaN,no_death,no_pd,0
2,3008,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,9,2011-06-01,V17,2021-03-01,9.749487,False,NaN,no_death,no_pd,0
3,3009,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,11,2011-06-01,V21,2025-06-01,14.001369,False,NaN,no_death,no_pd,0
4,3011,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,3,2011-07-01,V06,2013-09-01,2.171116,False,NaN,no_death,no_pd,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2242,425673,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,2,2024-11-01,V04,2026-04-01,1.412731,False,NaN,no_death,no_pd,0
2243,428231,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,2,2024-11-01,V04,2026-04-01,1.412731,False,NaN,no_death,no_pd,0
2244,430172,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,2,2025-01-01,V04,2026-04-01,1.245722,False,NaN,no_death,no_pd,0
2245,433261,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,2,2025-02-01,V04,2026-04-01,1.160849,False,NaN,no_death,no_pd,0


In [34]:
events["pd_case_type"].value_counts()

pd_case_type
no_pd                     2061
persistent_pd               88
pd_end_of_followup          74
pd_reversal                 20
pd_competing_diagnosis       4
Name: count, dtype: int64

### Modelling cohort (subste of at-risk based on modalities available)

In [33]:
cohort_events = events[
    events["PATNO"].isin(model_cohort["PATNO"])
].copy()

cohort_events

,PATNO,pd_event,pd_event_visit,pd_event_date,msa_event,msa_event_visit,msa_event_date,dlb_event,dlb_event_visit,dlb_event_date,n_visits,baseline_date,last_visit,last_visit_date,follow_up_years,death_event,death_date,death_timing,pd_case_type,pd_n_visits
0,3000,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,9,2011-02-01,V17,2021-05-01,10.245038,True,2026-03-01,death_without_pd,no_pd,0
1,3004,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,12,2011-04-01,V22,2026-05-01,15.082820,False,NaN,no_death,no_pd,0
2,3008,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,9,2011-06-01,V17,2021-03-01,9.749487,False,NaN,no_death,no_pd,0
4,3011,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,3,2011-07-01,V06,2013-09-01,2.171116,False,NaN,no_death,no_pd,0
5,3013,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,8,2011-11-01,V16,2021-03-01,9.330595,True,2023-06-01,death_without_pd,no_pd,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1353,219430,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,3,2023-03-01,V06,2025-04-01,2.086242,False,NaN,no_death,no_pd,0
1354,219562,True,V06,2025-05-01,False,NaN,NaN,False,NaN,NaN,3,2023-03-01,V06,2025-05-01,2.168378,False,NaN,no_death,pd_end_of_followup,1
1355,219621,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,4,2023-03-01,V08,2026-04-01,3.085558,False,NaN,no_death,no_pd,0
1365,220392,False,NaN,NaN,False,NaN,NaN,False,NaN,NaN,3,2023-05-01,V06,2025-06-01,2.086242,False,NaN,no_death,no_pd,0


In [36]:
# Create outcome col
pd_case_types = ["persistent_pd", "pd_end_of_followup"]

cohort_events["outcome"] = 0
cohort_events.loc[
    cohort_events["pd_case_type"].isin(pd_case_types),
    "outcome",
] = 1
cohort_events.loc[
    ~cohort_events["pd_case_type"].isin(pd_case_types)
    & cohort_events["death_event"],
    "outcome",
] = 2

cohort_events["outcome"].value_counts().sort_index()

outcome
0    673
1     55
2     24
Name: count, dtype: int64

Get outcome and time_to_event

In [38]:
# Convert date columns to datetime
for column in ["baseline_date", "pd_event_date", "death_date", "last_visit_date"]:
    cohort_events[column] = pd.to_datetime(cohort_events[column])

# Define the event date:
# 1 = PD, 2 = death, 0 = censored/event-free at last visit
cohort_events["event_date"] = cohort_events["last_visit_date"]

cohort_events.loc[
    cohort_events["outcome"] == 1,
    "event_date",
] = cohort_events.loc[
    cohort_events["outcome"] == 1,
    "pd_event_date",
]

cohort_events.loc[
    cohort_events["outcome"] == 2,
    "event_date",
] = cohort_events.loc[
    cohort_events["outcome"] == 2,
    "death_date",
]

cohort_events["time_to_event_days"] = (
    cohort_events["event_date"] - cohort_events["baseline_date"]
).dt.days

cohort_events["time_to_event_years"] = (
    cohort_events["time_to_event_days"] / 365.25
)

cohort_events[
    [
        "PATNO",
        "outcome",
        "baseline_date",
        "event_date",
        "time_to_event_days",
        "time_to_event_years",
    ]
].head()

,PATNO,outcome,baseline_date,event_date,time_to_event_days,time_to_event_years
0,3000,2,2011-02-01,2026-03-01,5507.0,15.077344
1,3004,0,2011-04-01,2026-05-01,5509.0,15.082820
2,3008,0,2011-06-01,2021-03-01,3561.0,9.749487
4,3011,0,2011-07-01,2013-09-01,793.0,2.171116
5,3013,2,2011-11-01,2023-06-01,4230.0,11.581109


In [46]:
cohort_events = cohort_events.dropna(
    subset=["time_to_event_years"]
).copy()



In [47]:
horizons = [2, 3, 4, 5]

for h in horizons:
    pd_events = ((cohort_events["outcome"] == 1) & (cohort_events["time_to_event_years"] <= h)).sum()
    deaths = ((cohort_events["outcome"] == 2) & (cohort_events["time_to_event_years"] <= h)).sum()
    
    print(f"{h} years: PD = {pd_events}, death = {deaths}")

2 years: PD = 20, death = 1
3 years: PD = 27, death = 2
4 years: PD = 40, death = 4
5 years: PD = 42, death = 5


In [48]:
cohort_events["censor_event"] = (cohort_events["outcome"] == 0).astype(int)
cohort_events.head()

,PATNO,pd_event,pd_event_visit,pd_event_date,msa_event,msa_event_visit,msa_event_date,dlb_event,dlb_event_visit,dlb_event_date,...,death_event,death_date,death_timing,pd_case_type,pd_n_visits,outcome,event_date,time_to_event_days,time_to_event_years,censor_event
0,3000,False,NaN,NaT,False,NaN,NaN,False,NaN,NaN,...,True,2026-03-01,death_without_pd,no_pd,0,2,2026-03-01,5507.0,15.077344,0
1,3004,False,NaN,NaT,False,NaN,NaN,False,NaN,NaN,...,False,NaT,no_death,no_pd,0,0,2026-05-01,5509.0,15.082820,1
2,3008,False,NaN,NaT,False,NaN,NaN,False,NaN,NaN,...,False,NaT,no_death,no_pd,0,0,2021-03-01,3561.0,9.749487,1
4,3011,False,NaN,NaT,False,NaN,NaN,False,NaN,NaN,...,False,NaT,no_death,no_pd,0,0,2013-09-01,793.0,2.171116,1
5,3013,False,NaN,NaT,False,NaN,NaN,False,NaN,NaN,...,True,2023-06-01,death_without_pd,no_pd,0,2,2023-06-01,4230.0,11.581109,0


In [49]:
from lifelines import KaplanMeierFitter

km_censor = KaplanMeierFitter()

km_censor.fit(
    durations=cohort_events["time_to_event_years"],
    event_observed=cohort_events["censor_event"]
)

for t in [2, 3, 4]:
    G_t = km_censor.predict(t)
    print(f"G({t}) = {G_t:.3f}")

G(2) = 0.946
G(3) = 0.856
G(4) = 0.612


In [60]:
t = cohort_events["time_to_event_years"]

print("0–<2 years:", ((t < 2) & (cohort_events["outcome"] == 0)).sum())
print("2–<3 years:", ((t >= 2) & (t < 3) & (cohort_events["outcome"] == 0)).sum())
print("3–<4 years:", ((t >= 3) & (t < 4) & (cohort_events["outcome"] == 0)).sum())
print("4–<5 years:", ((t >= 4) & (t < 5) & (cohort_events["outcome"] == 0)).sum())
print("5+ years:", ((t >= 5) & (cohort_events["outcome"] == 0)).sum())

0–<2 years: 40
2–<3 years: 65
3–<4 years: 152
4–<5 years: 65
5+ years: 351


In [61]:
# Why did many people drop off between 3 - 4 years
cohort_events.loc[
    (cohort_events["outcome"] == 0) &
    (cohort_events["time_to_event_years"] >= 3) &
    (cohort_events["time_to_event_years"] < 4),
    "time_to_event_years"
].describe()

count    152.000000
mean       3.168090
std        0.220076
min        3.000684
25%        3.000684
50%        3.085558
75%        3.170431
max        3.923340
Name: time_to_event_years, dtype: float64

In [62]:
cohort_events["outcome"].value_counts().sort_index()

outcome
0    673
1     55
2     23
Name: count, dtype: int64

In [65]:
for t in [2, 3, 4, 5]:
    n_at_risk = (cohort_events["time_to_event_years"] >= t).sum()
    print(f"{t} years: {n_at_risk} at risk")

print(len(cohort_events))
cohort_events["outcome"].value_counts(dropna=False)

2 years: 690 at risk
3 years: 617 at risk
4 years: 451 at risk
5 years: 382 at risk
751


outcome
0    673
1     55
2     23
Name: count, dtype: int64

In [43]:
# Check that one person that has no event_date

duration_col = cohort_events["time_to_event_years"]
event_col = cohort_events["censor_event"]

print("NaNs in durations:", duration_col.isna().sum())
print("NaNs in event_observed:", event_col.isna().sum())

cohort_events.loc[
    duration_col.isna() | event_col.isna(),
    ["PATNO", "outcome", "event_date", "time_to_event_years", "censor_event"],
]

NaNs in durations: 1
NaNs in event_observed: 0


,PATNO,outcome,event_date,time_to_event_years,censor_event
174,3969,2,NaT,NaN,0


In [7]:
# Restrict events to participants in the model cohort and selected PD case types
model_events = events[
    events["PATNO"].isin(model_cohort["PATNO"])
    & events["pd_case_type"].isin(["persistent_pd", "pd_end_of_followup"])
].copy()

# Summarize total participants and those with follow-up <= 2 years
summary = (
    model_events
    .groupby("pd_case_type")
    .agg(
        total_participants=("PATNO", "nunique"),
        participants_follow_up_le_2_years=(
            "follow_up_years",
            lambda x: (x <= 2).sum()
        ),
    )
    .reset_index()
)

summary

,pd_case_type,total_participants,participants_follow_up_le_2_years
0,pd_end_of_followup,15,1
1,persistent_pd,40,1


In [8]:
short_follow_up_participants = (
    model_events.loc[
        model_events["follow_up_years"] < 2,
        ["PATNO", "pd_case_type", "follow_up_years", "pd_event_date"]
    ]
    .sort_values("follow_up_years")
)

print(short_follow_up_participants.to_string(index=False))

 PATNO       pd_case_type  follow_up_years pd_event_date
 52362 pd_end_of_followup         1.081451    2017-05-01
217775      persistent_pd         1.924709    2024-04-01


## Confirmed PD

In [9]:
# Calculate time from baseline to PD event
model_events["baseline_date"] = pd.to_datetime(model_events["baseline_date"])
model_events["pd_event_date"] = pd.to_datetime(model_events["pd_event_date"])

model_events["time_to_event_days"] = (
    model_events["pd_event_date"] - model_events["baseline_date"]
).dt.days

model_events["time_to_event_years"] = (
    model_events["time_to_event_days"] / 365.25
)

confirmed_pd = model_events[
    ["PATNO", "pd_case_type", "baseline_date", "pd_event_date", "time_to_event_days", "time_to_event_years"]
].sort_values("time_to_event_years")
len(confirmed_pd)
confirmed_pd

,PATNO,pd_case_type,baseline_date,pd_event_date,time_to_event_days,time_to_event_years
742,113351,persistent_pd,2021-11-01,2022-06-01,212,0.580424
405,56169,persistent_pd,2017-01-01,2018-01-01,365,0.999316
627,75564,persistent_pd,2019-01-01,2020-01-01,365,0.999316
199,14426,persistent_pd,2014-02-01,2015-03-01,393,1.075975
1249,201871,persistent_pd,2023-02-01,2024-03-01,394,1.078713
1054,174433,persistent_pd,2022-11-01,2023-12-01,395,1.081451
334,52362,pd_end_of_followup,2016-04-01,2017-05-01,395,1.081451
377,54215,persistent_pd,2016-07-01,2017-08-01,396,1.084189
1048,173622,persistent_pd,2022-12-01,2024-01-01,396,1.084189
814,134566,persistent_pd,2022-01-01,2023-02-01,396,1.084189


In [10]:
confirmed_pd["time_to_event_years"].agg(["mean", "std", "median"])

mean      3.548504
std       2.707676
median    3.000684
Name: time_to_event_years, dtype: float64

In [31]:
def summarize_pd_events(horizon):
    """Print PD event counts and proportions for a specified time horizon."""
    time_to_event = confirmed_pd["time_to_event_years"]

    print("PD events:", len(confirmed_pd))
    print(f"PD <= {horizon} years:", (time_to_event <= horizon).sum())
    print(f"PD > {horizon} years:", (time_to_event > horizon).sum())
    print(
        f"Proportion within {horizon} years:",
        (time_to_event <= horizon).mean(),
    )

summarize_pd_events(5)

PD events: 55
PD <= 5 years: 42
PD > 5 years: 13
Proportion within 5 years: 0.7636363636363637


## Death 

In [12]:
model_death_events = events[
    events["PATNO"].isin(model_cohort["PATNO"])
    & events["death_event"]
    & ~events["pd_event"]
].copy()

model_death_events["baseline_date"] = pd.to_datetime(
    model_death_events["baseline_date"]
)
model_death_events["death_date"] = pd.to_datetime(
    model_death_events["death_date"]
)

model_death_events["time_to_death_days"] = (
    model_death_events["death_date"]
    - model_death_events["baseline_date"]
).dt.days

model_death_events["time_to_death_years"] = (
    model_death_events["time_to_death_days"] / 365.25
)

death_events = model_death_events[
    [
        "PATNO",
        "baseline_date",
        "death_date",
        "time_to_death_days",
        "time_to_death_years",
    ]
].sort_values("time_to_death_years")

print("Deaths without PD:", len(death_events))
death_events

Deaths without PD: 24


,PATNO,baseline_date,death_date,time_to_death_days,time_to_death_years
338,52515,2016-03-01,2017-08-01,518.0,1.418207
84,3450,2010-11-01,2013-08-01,1004.0,2.748802
834,138762,2023-03-01,2026-03-01,1096.0,3.000684
387,54915,2016-09-01,2019-12-01,1186.0,3.247091
594,74375,2018-09-01,2023-09-01,1826.0,4.999316
170,3965,2013-02-01,2018-11-01,2099.0,5.746749
293,50081,2017-04-01,2023-04-01,2191.0,5.998631
103,3526,2012-01-01,2018-02-01,2223.0,6.086242
488,60006,2014-09-01,2021-04-01,2404.0,6.581793
635,92834,2015-02-01,2022-12-01,2860.0,7.830253


In [15]:

print(
    "Deaths:",
    len(death_events)
)

print(
    "Deaths <= 2 years:",
    (death_events["time_to_death_years"] <= 3).sum()
)

print(
    "Deaths > 2 years:",
    (death_events["time_to_death_years"] > 3).sum()
)

Deaths: 24
Deaths <= 2 years: 2
Deaths > 2 years: 21


In [32]:
def summarize_deaths(horizon):
    """Print death counts relative to a specified time horizon."""
    time_to_death = death_events["time_to_death_years"]

    print("Deaths:", len(death_events))
    print(f"Deaths <= {horizon} years:", (time_to_death <= horizon).sum())
    print(f"Deaths > {horizon} years:", (time_to_death > horizon).sum())


summarize_deaths(5)

Deaths: 24
Deaths <= 5 years: 5
Deaths > 5 years: 18
